In [ ]:
import sys


print(f"Installing scikit-learn into: {sys.executable}")
!{sys.executable} -m pip install scikit-learn pandas numpy

print("\nInstallation complete. Now try running your imports again.")

Installing scikit-learn into: c:\Users\PMLS\AppData\Local\Programs\Python\Python313\python.exe
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.7 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/8.7 MB 3.3 MB/s eta 0:00:03
   ------------- -------------------------- 2.9/8.7 MB 5.3 MB/s eta 0:00:02
   -------------- ------------------------- 3.1/8.7 MB 5.5 MB/s eta 0:00:02
   ---------------- ----------------------- 3.7/8.7 MB 4.1 MB/s eta 0:00:02
   ------------------ --------------------- 3.9/8.7 MB 3.8 MB/s eta 0:00:02
   -------------------- ------------------- 4.5/8.7 MB 3.6 MB/s eta 0:00:02
   ------------------------------ --------- 6.6/8.7 MB 4.0 MB/s eta 0:00:01
   ---------------------------------- ----- 7.6/8.7 MB 4.3 MB/s eta 0:00:01
   ------------------


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import os
import json
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")

Libraries imported successfully.


In [8]:

# ==========================================
# CONFIGURATION & LOADING
# ==========================================

# We use r"..." (raw string) to handle Windows backslashes correctly
HUMAN_DATA_PATH = r"C:\Users\PMLS\OneDrive\Desktop\ai_sem_proj_semeval-2026-task-4-baselines\data\raw\dev_track_a.jsonl"
SYNTHETIC_DATA_PATH = r"C:\Users\PMLS\OneDrive\Desktop\ai_sem_proj_semeval-2026-task-4-baselines\data\processed\combined_synthetic_for_training.jsonl"

print(f"Human Data Path: {HUMAN_DATA_PATH}")
print(f"Synthetic Data Path: {SYNTHETIC_DATA_PATH}") 

def load_data(filepath):
    """
    Reads a JSONL file and returns a pandas DataFrame.
    """
    # Check if file exists before trying to load
    if not os.path.exists(filepath):
        print(f"CRITICAL ERROR: File still not found at: {filepath}")
        return None
        
    print(f"Loading data from: {filepath}")
    return pd.read_json(filepath, lines=True)

Human Data Path: C:\Users\PMLS\OneDrive\Desktop\ai_sem_proj_semeval-2026-task-4-baselines\data\raw\dev_track_a.jsonl
Synthetic Data Path: C:\Users\PMLS\OneDrive\Desktop\ai_sem_proj_semeval-2026-task-4-baselines\data\processed\combined_synthetic_for_training.jsonl


In [9]:
def load_data(filepath):
    """
    Reads a JSONL file and returns a pandas DataFrame.
    """
    if not os.path.exists(filepath):
        print(f"Error: File not found at {filepath}")
        return None
        
    print(f"Loading data from: {filepath}")
    # using lines=True for jsonl files
    return pd.read_json(filepath, lines=True)

def preprocess_data(df):
    """
    Universal Preprocessor: Handles both Human (dev_track_a) 
    and Synthetic (anchor_story/similar_story) formats.
    """
    texts = []
    labels = []
    
    for index, row in df.iterrows():
        story = ""
        opt1 = ""
        opt2 = ""
        label = 0
        
        # -----------------------------------------------
        # CASE 1: Human Data (dev_track_a.jsonl)
        # -----------------------------------------------
        if 'anchor_text' in row and pd.notna(row['anchor_text']):
            story = row['anchor_text']
            opt1 = row['text_a']
            opt2 = row['text_b']
            
            # Logic: If text_a is closer, label is 0. If false, label is 1.
            if row['text_a_is_closer'] is True:
                label = 0
            else:
                label = 1

        # -----------------------------------------------
        # CASE 2: Synthetic Data (anchor_story, similar_story)
        # -----------------------------------------------
        elif 'anchor_story' in row and pd.notna(row['anchor_story']):
            story = row['anchor_story']
            
            # The synthetic data gives us the explicit answer.
            # similar_story = Correct
            # dissimilar_story = Incorrect
            
            # To prevent the model from learning "Option A is always correct",
            # we will randomly flip them for training.
            if random.random() > 0.5:
                # Option A is correct
                opt1 = row['similar_story']
                opt2 = row['dissimilar_story']
                label = 0
            else:
                # Option B is correct (we flip them)
                opt1 = row['dissimilar_story']
                opt2 = row['similar_story']
                label = 1
                
        # -----------------------------------------------
        # Combine and Save
        # -----------------------------------------------
        if story:
            full_text = f"Story: {story} Option A: {opt1} Option B: {opt2}"
            texts.append(full_text)
            labels.append(label)

    return texts, labels

In [10]:
# Load the datasets
df_human = load_data(HUMAN_DATA_PATH)
df_synth = load_data(SYNTHETIC_DATA_PATH)

# Check if data loaded correctly
if df_human is not None and df_synth is not None:
    # Combine the dataframes
    df_train_full = pd.concat([df_human, df_synth], ignore_index=True)
    
    print(f"\nHuman samples: {len(df_human)}")
    print(f"Synthetic samples: {len(df_synth)}")
    print(f"Total Combined Training Data: {len(df_train_full)}")
    
    # Show the first few rows to verify structure
    display(df_train_full.head(2))
else:
    print("Failed to load data. Please check paths.")

Loading data from: C:\Users\PMLS\OneDrive\Desktop\ai_sem_proj_semeval-2026-task-4-baselines\data\raw\dev_track_a.jsonl
Loading data from: C:\Users\PMLS\OneDrive\Desktop\ai_sem_proj_semeval-2026-task-4-baselines\data\processed\combined_synthetic_for_training.jsonl

Human samples: 200
Synthetic samples: 2397
Total Combined Training Data: 2597


,anchor_text,text_a,text_b,text_a_is_closer,anchor_story,similar_story,dissimilar_story,source
0,The book follows an international organization...,The old grandmother Tina arrives in town to at...,The nano-plague that poisoned Earth's water su...,False,NaN,NaN,NaN,NaN
1,"Glenn Tyler (Elvis Presley), a childish 25-yea...","Bill Babbitt supported the death penalty, unti...",A white-collar suburban father Kyle (Fran Kran...,True,NaN,NaN,NaN,NaN


In [11]:
# Prepare the lists of text and labels
X, y = preprocess_data(df_train_full)

# Split into Train and Validation sets (80/20 split)
# This allows us to measure accuracy before submitting
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")

# Create the Pipeline
# 1. TF-IDF: Convert text to numbers, removing common english words
# 2. LogisticRegression: The classifier
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000)),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
])

# Train the model
print("\nTraining the model... (this may take a moment)")
pipeline.fit(X_train, y_train)
print("Training complete.")

Training set size: 2077
Validation set size: 520

Training the model... (this may take a moment)
Training complete.


In [12]:
# Predict on the validation set
predictions = pipeline.predict(X_val)

# Calculate metrics
acc = accuracy_score(y_val, predictions)

print(f"Validation Accuracy: {acc:.4f}")
print("\nDetailed Classification Report:")
print(classification_report(y_val, predictions))

Validation Accuracy: 0.4846

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.49      0.49      0.49       264
           1       0.48      0.48      0.48       256

    accuracy                           0.48       520
   macro avg       0.48      0.48      0.48       520
weighted avg       0.48      0.48      0.48       520

